# Practitioner Analytics & OOS Holdout

Performance (§2.1–2.3) → Risk (§2.4–2.6) → Implementation (§2.7–2.8) → Allocations (§2.9–2.11) → Cross-strategy structure (§2.12) → Robustness (§2.13–2.14) → Sensitivity (§2.15–2.16) → OOS Holdout (Part 3)

In [ ]:
import sys
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import scipy.stats as stats
import warnings
warnings.filterwarnings("ignore")

from _shared import (
    ROOT, FAMILY_COLORS, FAMILY_MAP, FAMILY_ORDER, DISPLAY,
    load_base_returns, load_vmp_returns, load_regime_signals,
    load_weights_cache, build_all_stats, build_switch_v2a,
    ann_sharpe, ann_return, ann_vol, max_drawdown, CRISES
)

matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

base = load_base_returns()
vmp  = load_vmp_returns()
sw_oos = pd.read_parquet(ROOT / "data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet")
canon = pd.read_csv(ROOT / "data/cache/appendix_a_canonical.csv")
regime_sig = load_regime_signals()

switch_v2a = build_switch_v2a(base, sw_oos)

print(f"Base: {base.shape}, VMP: {vmp.shape}, canon: {len(canon)} rows")

## Chapter Introduction

Notebook 01 established **which** families win on the 62-strategy harness — the ranking, the VMP-lift universality, and the statistical significance of the key contrasts. This chapter asks how they **behave**: through time, in crises, under realistic costs, and — critically — whether the full-sample findings survive on data the strategies never saw. The universe is 29 assets spanning 2003–2026 (~23 years, three major stress episodes). This is Paper 1's practitioner and robustness layer: every section answers a question a portfolio manager would ask before allocating capital.

---
## Performance

A full-sample Sharpe hides *when* a strategy earned its keep. The rolling and calendar-year views below distinguish consistent compounders from strategies that look good in aggregate but are streaky or crisis-dependent. The 5-year bucket table (§2.3) makes the temporal stability caveat concrete: sub-period Sharpes vary far more than the cross-family ranking gap.

## §2.1 Rolling 12-Month Sharpe

In [ ]:
ROLLING_STRATS = [
    ("EW",                  "EW",              base,  "EW (benchmark)"),
    ("GMV(ledoit_wolf)",    "GMV(LW)",         base,  "Classical MV"),
    ("MDP(ledoit_wolf)",    "MDP(LW)",         base,  "Diversification"),
    ("MSR(ledoit_wolf)",    "MSR(LW)",         base,  "Classical MV"),
    ("SWITCH(ledoit_wolf)", "SWITCH(LW)",      base,  "Regime Switch"),
    ("VMP(MDP(ledoit_wolf))", "VMP(MDP(LW))", vmp,   "Diversification"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True)
axes_flat = axes.flatten()

for idx, (col, title, src, family) in enumerate(ROLLING_STRATS):
    ax = axes_flat[idx]
    color = FAMILY_COLORS.get(family, "#aaaaaa")
    rets = src[col] if col in src.columns else None
    if rets is None:
        ax.text(0.5, 0.5, f"{title}\nnot found", ha="center", va="center", transform=ax.transAxes)
        continue
    roll_s = rets.rolling(252).mean() / rets.rolling(252).std() * np.sqrt(252)
    for start, end, _ in CRISES:
        ax.axvspan(start, end, color="grey", alpha=0.12, zorder=0)
    ax.plot(roll_s.index, roll_s.values, color=color, lw=1.2, zorder=3)
    ax.axhline(1.0, color="black", ls="--", lw=0.8, alpha=0.6)
    ax.axhline(0.0, color="gray", ls=":", lw=0.6, alpha=0.4)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_ylabel("Rolling Sharpe (252d)", fontsize=8)
    ax.tick_params(axis="x", rotation=30, labelsize=7)

fig.suptitle("§2.1 Rolling 12-Month Sharpe — 6 Key Strategies (2003–2026)", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

## §2.2 Calendar-Year Returns Heatmap

24 base strategies × 24 calendar years (2003–2026, with 2026 partial through April).

In [ ]:
CAL_STRATS = [
    ("EW", "EW"),
    ("GMV(sample)", "GMV(samp)"),
    ("GMV(ledoit_wolf)", "GMV(LW)"),
    ("MSR(sample)", "MSR(samp)"),
    ("MSR(ledoit_wolf)", "MSR(LW)"),
    ("MDP(sample)", "MDP(samp)"),
    ("MDP(ledoit_wolf)", "MDP(LW)"),
    ("RP(sample)", "RP(samp)"),
    ("RP(ledoit_wolf)", "RP(LW)"),
    ("HRP(sample)", "HRP(samp)"),
    ("HRP(ledoit_wolf)", "HRP(LW)"),
    ("SWITCH(sample)", "SWITCH(samp)"),
    ("SWITCH(ledoit_wolf)", "SWITCH(LW)"),
    ("TSMOM(12m)", "TSMOM(12m)"),
    ("TSMOM(6m)", "TSMOM(6m)"),
    ("BL-Eq(sample)", "BL-Eq(samp)"),
    ("BL-Eq(LW)", "BL-Eq(LW)"),
    ("BL-Mom(LW)", "BL-Mom(LW)"),
    ("BL-Rev(LW)", "BL-Rev(LW)"),
    ("FF3-Mom", "FF3-Mom"),
    ("FF3-LowVol", "FF3-LowVol"),
    ("FF3-Quality", "FF3-Qlty"),
    ("FF3-Multi", "FF3-Multi"),
]

years = list(range(2003, 2027))
cal_labels = [str(y) if y < 2026 else "2026*" for y in years]

cal_rows = {}
for col, disp in CAL_STRATS:
    if col not in base.columns:
        continue
    r = base[col]
    cal_rows[disp] = [
        (1 + r[r.index.year == y]).prod() - 1 if (r.index.year == y).any() else np.nan
        for y in years
    ]
cal_rows["SWITCH(v2a)"] = [
    (1 + switch_v2a[switch_v2a.index.year == y]).prod() - 1
    if (switch_v2a.index.year == y).any() else np.nan
    for y in years
]

df_cal = pd.DataFrame(cal_rows, index=cal_labels).T
vmax_cal = 0.50
Z_cal = df_cal.values.clip(-vmax_cal, vmax_cal)

fig, ax = plt.subplots(figsize=(18, 12))
im = ax.imshow(Z_cal, cmap="RdYlGn", vmin=-vmax_cal, vmax=vmax_cal, aspect="auto")
plt.colorbar(im, ax=ax, label="Annual Return",
             format=FuncFormatter(lambda x, _: f"{x:.0%}"), shrink=0.75)

nstrats, nyears = Z_cal.shape
for i in range(nstrats):
    for j in range(nyears):
        val = df_cal.iloc[i, j]
        if not np.isnan(val):
            brightness = (val + vmax_cal) / (2 * vmax_cal)
            tc = "white" if brightness < 0.25 or brightness > 0.75 else "black"
            ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=5, color=tc)

ax.set_xticks(range(nyears))
ax.set_xticklabels(cal_labels, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(nstrats))
ax.set_yticklabels(df_cal.index.tolist(), fontsize=7.5)
ax.set_title("§2.2 Calendar-Year Returns — 24 Strategies, 2003–2026 (* 2026 through April)",
             fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()

## §2.3 Sub-Period Sharpe (5-Year Buckets)

Full-sample Sharpe is a single number over 23 years. This table slices the same window into five consecutive 5-year buckets to expose regime sensitivity: which strategies were rewarded in the GFC decade, which stumbled in the rate-shock era, and which earned their full-sample rank consistently across all five windows.

In [ ]:
from IPython.display import display

BUCKETS = [
    ("2003–2007", "2003-01-01", "2007-12-31"),
    ("2008–2012", "2008-01-01", "2012-12-31"),
    ("2013–2017", "2013-01-01", "2017-12-31"),
    ("2018–2022", "2018-01-01", "2022-12-31"),
    ("2023–2026", "2023-01-01", "2026-12-31"),
]

REP_STRATS = [
    ("EW",                    base),
    ("GMV(ledoit_wolf)",      base),
    ("MSR(ledoit_wolf)",      base),
    ("MDP(ledoit_wolf)",      base),
    ("SWITCH(ledoit_wolf)",   base),
    ("BL-Mom(LW)",            base),
    ("VMP(MDP(ledoit_wolf))", vmp),
]

rows = {}
for label, start, end in BUCKETS:
    col_sharpes = {}
    for col, src in REP_STRATS:
        r = src[col].loc[start:end].dropna() if col in src.columns else pd.Series(dtype=float)
        col_sharpes[col.replace("ledoit_wolf", "LW")] = ann_sharpe(r) if len(r) > 21 else np.nan
    sv = switch_v2a.loc[start:end].dropna()
    col_sharpes["SWITCH(v2a)"] = ann_sharpe(sv) if len(sv) > 21 else np.nan
    rows[label] = col_sharpes

df_subperiod = pd.DataFrame(rows).T.round(3)
display(df_subperiod)


---
## Risk

Risk-adjusted ranking says nothing about the depth of the holes a strategy forces investors to sit through. Underwater paths through the GFC, COVID crash, and 2022 rate-shock bear market reveal which families protect capital when it matters most, and CVaR quantifies the expected loss conditional on being in the worst 5% of days.

## §2.4 Underwater Drawdown

In [ ]:
UW_STRATS = [
    ("EW",                  "EW",            base, "EW (benchmark)"),
    ("MDP(ledoit_wolf)",    "MDP(LW)",       base, "Diversification"),
    ("MSR(ledoit_wolf)",    "MSR(LW)",       base, "Classical MV"),
    ("SWITCH(ledoit_wolf)", "SWITCH(LW)",    base, "Regime Switch"),
    ("VMP(MDP(ledoit_wolf))", "VMP(MDP(LW))", vmp, "Diversification"),
]

fig, ax = plt.subplots(figsize=(14, 6))

for col, label, src, family in UW_STRATS:
    color = FAMILY_COLORS.get(family, "#aaaaaa")
    rets = src[col].dropna() if col in src.columns else None
    if rets is None:
        continue
    cum = (1 + rets).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax() * 100
    ax.fill_between(dd.index, dd.values, 0, alpha=0.10, color=color)
    ax.plot(dd.index, dd.values, color=color, label=label, lw=1.4, alpha=0.7)

# SWITCH(v2a)
rets_sv2a = switch_v2a.dropna()
cum_sv2a = (1 + rets_sv2a).cumprod()
dd_sv2a = (cum_sv2a - cum_sv2a.cummax()) / cum_sv2a.cummax() * 100
ax.fill_between(dd_sv2a.index, dd_sv2a.values, 0, alpha=0.10, color=FAMILY_COLORS["Regime Switch"])
ax.plot(dd_sv2a.index, dd_sv2a.values, color=FAMILY_COLORS["Regime Switch"],
        label="SWITCH(v2a)", lw=1.8, ls="--", alpha=0.7)

for start, end, _ in CRISES:
    ax.axvspan(start, end, color="grey", alpha=0.08, zorder=0)

ax.set_xlabel("Date")
ax.set_ylabel("Drawdown (%)")
ax.set_title("§2.4 Underwater Drawdown — 6 Strategies (2003–2026)", fontsize=11, fontweight="bold")
ax.legend(ncol=3, fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

## §2.5 CVaR(95%) Bar Chart

In [ ]:
def cvar_95(r):
    r = r.dropna()
    threshold = np.percentile(r, 5)
    return r[r <= threshold].mean() * np.sqrt(252)  # annualized

cvar_records = []
for col in base.columns:
    fam = FAMILY_MAP.get(col, "Classical MV")
    cvar_records.append({"display": col.replace("ledoit_wolf", "LW"), "family": fam,
                          "cvar": cvar_95(base[col]), "is_vmp": False})
for col in vmp.columns:
    base_col = col[4:-1]
    fam = FAMILY_MAP.get(base_col, "Classical MV")
    cvar_records.append({"display": col.replace("ledoit_wolf", "LW"), "family": fam,
                          "cvar": cvar_95(vmp[col]), "is_vmp": True})

df_cvar = pd.DataFrame(cvar_records).sort_values("cvar")

fig, ax = plt.subplots(figsize=(10, 14))
colors = [FAMILY_COLORS.get(f, "#aaa") for f in df_cvar["family"]]
bars = ax.barh(range(len(df_cvar)), df_cvar["cvar"], color=colors, alpha=0.8)
ax.set_yticks(range(len(df_cvar)))
ax.set_yticklabels(df_cvar["display"], fontsize=6)
ax.set_xlabel("CVaR(95%) — Annualized")
ax.set_title("§2.5 CVaR(95%) — All 62 Strategies (sorted)", fontsize=11, fontweight="bold")
patches = [mpatches.Patch(color=FAMILY_COLORS.get(f, "#aaa"), label=f)
           for f in sorted(df_cvar["family"].unique())]
ax.legend(handles=patches, ncol=2, fontsize=7, loc="lower right")
ax.axvline(0, color="gray", lw=0.7, ls="--", alpha=0.5)
fig.tight_layout()
plt.show()

## §2.6 Rolling 252-Day Correlation to SPY

6 representative strategies vs SPY.US returns. Higher correlation = more equity beta exposure.

In [ ]:
prices_29 = pd.read_parquet(ROOT / "data/cache/prices_29.parquet")
spy_ret = prices_29["SPY.US"].pct_change().dropna()

CORR_STRATS = [
    ("GMV(ledoit_wolf)",      "GMV(LW)",        base,  "Classical MV"),
    ("MDP(ledoit_wolf)",      "MDP(LW)",         base,  "Diversification"),
    ("MSR(ledoit_wolf)",      "MSR(LW)",         base,  "Classical MV"),
    ("SWITCH(ledoit_wolf)",   "SWITCH(LW)",      base,  "Regime Switch"),
    ("VMP(MDP(ledoit_wolf))", "VMP(MDP(LW))",    vmp,   "Diversification"),
    ("BL-Mom(LW)",            "BL-Mom(LW)",      base,  "Black-Litterman"),
]

spy_aligned = spy_ret.reindex(base.index).ffill()

fig, ax = plt.subplots(figsize=(16, 9))
for col, label, src, family in CORR_STRATS:
    color = FAMILY_COLORS.get(family, "#aaa")
    rets = src[col] if col in src.columns else None
    if rets is None:
        continue
    combined = pd.concat([spy_aligned, rets], axis=1).dropna()
    roll_corr = combined.iloc[:, 0].rolling(252).corr(combined.iloc[:, 1])
    ax.plot(roll_corr.index, roll_corr.values, label=label, color=color, lw=1.2)

for start, end, _ in CRISES:
    ax.axvspan(start, end, color="grey", alpha=0.10, zorder=0)
ax.axhline(1.0, color="gray", ls="--", lw=0.7, alpha=0.5)
ax.set_xlabel("Date")
ax.set_ylabel("Rolling 252-Day Correlation to SPY")
ax.set_title("§2.6 Rolling Correlation to SPY — 6 Strategies", fontsize=11, fontweight="bold")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

---
## Implementation

A backtest Sharpe is gross — a practitioner pays to trade. Turnover determines how much of each strategy's edge survives at realistic transaction costs, and the concentration diagnostics below reveal whether the portfolio is actually diversified or quietly hiding a single-name bet.

## §2.7 Turnover Histogram + Boxplot

In [ ]:
canon_num = canon.copy()
canon_num["turnover"] = pd.to_numeric(canon_num["turnover"], errors="coerce")
to_data = canon_num["turnover"].dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(to_data, bins=20, color="#1f77b4", alpha=0.75, edgecolor="white")
ax1.set_xlabel("Average Daily Turnover (%)")
ax1.set_ylabel("Count")
ax1.set_title("§2.7 Turnover Histogram — All 62 Strategies", fontweight="bold")
ax1.axvline(to_data.median(), color="red", ls="--", lw=1.0, label=f"Median={to_data.median():.2f}%")
ax1.legend(fontsize=8)

by_family = [
    canon_num[canon_num["family"] == fam]["turnover"].dropna().values
    for fam in sorted(canon_num["family"].unique())
]
fam_labels = sorted(canon_num["family"].unique())
bp = ax2.boxplot(by_family, vert=True, patch_artist=True)
for patch, fam in zip(bp["boxes"], fam_labels):
    patch.set_facecolor(FAMILY_COLORS.get(fam, "#aaa"))
    patch.set_alpha(0.7)
ax2.set_xticks(range(1, len(fam_labels) + 1))
ax2.set_xticklabels(fam_labels, rotation=30, ha="right", fontsize=7)
ax2.set_ylabel("Turnover (%)")
ax2.set_title("Turnover by Family", fontweight="bold")
fig.tight_layout()
plt.show()

## §2.8 Maximum Single-Position Weight Distribution

29-asset portfolio.

In [ ]:
from _shared import WEIGHTS_SUFFIX

WEIGHT_FILE_MAP = {
    "EW":                "EW",
    "GMV(sample)":       "GMV_sample",
    "GMV(ledoit_wolf)":  "GMV_ledoit_wolf",
    "MDP(sample)":       "MDP_sample",
    "MDP(ledoit_wolf)":  "MDP_ledoit_wolf",
    "MSR(sample)":       "MSR_sample",
    "MSR(ledoit_wolf)":  "MSR_ledoit_wolf",
    "HRP(ledoit_wolf)":  "HRP_ledoit_wolf",
    "SWITCH(ledoit_wolf)": "SWITCH_ledoit_wolf",
    "BL-Mom(LW)":        "BL-Mom_LW",
    "FF3-LowVol":        "FF3-LowVol",
}

max_weight_records = []
for col, stem in WEIGHT_FILE_MAP.items():
    w = load_weights_cache(stem)
    if w is not None:
        max_w = w.max(axis=1).dropna()
        max_weight_records.append({"strategy": col.replace("ledoit_wolf", "LW"), "max_w": max_w.mean()})

df_mw = pd.DataFrame(max_weight_records).sort_values("max_w")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_mw["strategy"], df_mw["max_w"], color="#2ca02c", alpha=0.75)
ax.set_xlabel("Mean Max Single-Asset Weight")
ax.set_title("§2.8 Average Maximum Single-Position Weight — 29-asset portfolio", fontweight="bold")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0%}"))
fig.tight_layout()
plt.show()

---
## Allocations

What is each family actually doing with the money? The pie charts and timelines below show average and time-varying asset-class exposure; the crisis snapshots reveal how allocations shift precisely when regime breaks impose the largest drawdowns.

## §2.9 Average Weight Pie Charts — 6 Strategies by Asset Class

In [ ]:
ASSET_CLASSES = {
    "US Single Stocks":  ["AAPL.US", "MSFT.US", "GOOGL.US", "NVDA.US",
                           "JPM.US", "JNJ.US", "XOM.US", "WMT.US"],
    "US Sector ETFs":    ["XLK.US", "XLF.US", "XLE.US", "XLV.US", "XLP.US", "XLU.US"],
    "Broad Equity ETFs": ["SPY.US", "IWM.US"],
    "Intl Equity ETFs":  ["EFA.US", "EEM.US", "FXI.US"],
    "Fixed Income":      ["SHY.US", "IEF.US", "TLT.US", "AGG.US", "HYG.US"],
    "Commodities+FX":    ["GLD.US", "SLV.US", "DBC.US", "USO.US", "EURUSD"],
}
AC_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#d62728", "#8c564b"]
ac_labels = list(ASSET_CLASSES.keys())

PIE_STRATS = [
    ("EW",                 "EW",          "EW"),
    ("GMV(LW)",            "GMV(LW)",     "GMV_ledoit_wolf"),
    ("MSR(LW)",            "MSR(LW)",     "MSR_ledoit_wolf"),
    ("MDP(LW)",            "MDP(LW)",     "MDP_ledoit_wolf"),
    ("SWITCH(LW)",         "SWITCH(LW)",  "SWITCH_ledoit_wolf"),
    ("BL-Mom(LW)",         "BL-Mom(LW)", "BL-Mom_LW"),
]

fig, axes = plt.subplots(3, 2, figsize=(12, 16))
axes_flat = axes.flatten()

for idx, (key, title, stem) in enumerate(PIE_STRATS):
    ax = axes_flat[idx]
    w = load_weights_cache(stem)
    if w is None:
        ax.text(0.5, 0.5, f"{title}\nno weights", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title, fontweight="bold")
        continue
    mean_w = w.mean()
    ac_vals = []
    for ac, assets in ASSET_CLASSES.items():
        cols = [a for a in assets if a in mean_w.index]
        ac_vals.append(mean_w[cols].sum() if cols else 0.0)
    total = sum(ac_vals)
    if total > 0:
        ac_vals = [v / total for v in ac_vals]
    wedges, texts, autotexts = ax.pie(
        ac_vals, colors=AC_COLORS, startangle=90,
        wedgeprops={"linewidth": 0.5, "edgecolor": "white"},
        autopct=lambda p: f"{p:.1f}%" if p >= 2.5 else "", pctdistance=0.80,
    )
    for t in texts:
        t.set_fontsize(9)
    for at in autotexts:
        at.set_fontsize(10)
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)

patches = [mpatches.Patch(color=AC_COLORS[i], label=ac_labels[i]) for i in range(len(ac_labels))]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("§2.9 Average Weight by Asset Class — 6 Strategies (2003–2026)",
             fontsize=12, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

## §2.10 Asset-Class Allocation Timeline

In [ ]:
from IPython.display import Image
Image("../docs/figures/asset_class_allocation_timeline.png")

## §2.11 Crisis-Date Allocation Snapshots

In [ ]:
CRISIS_DATES = [
    ("2008-09-15", "GFC (Sep 15 2008)"),
    ("2020-03-20", "COVID (Mar 20 2020)"),
    ("2022-06-15", "Rate Shock (Jun 15 2022)"),
]
SNAP_STRATS = [
    ("EW",         "EW"),
    ("MDP(LW)",    "MDP_ledoit_wolf"),
    ("SWITCH(LW)", "SWITCH_ledoit_wolf"),
]

fig, axes = plt.subplots(len(SNAP_STRATS), len(CRISIS_DATES),
                          figsize=(14, 9), sharey="row")

for row_idx, (strat_label, stem) in enumerate(SNAP_STRATS):
    w = load_weights_cache(stem)
    for col_idx, (date_str, date_label) in enumerate(CRISIS_DATES):
        ax = axes[row_idx, col_idx]
        if w is None:
            ax.text(0.5, 0.5, "no weights", ha="center", va="center", transform=ax.transAxes)
            continue
        dt = pd.Timestamp(date_str)
        # Find nearest available date
        idx_pos = w.index.get_indexer([dt], method="nearest")[0]
        snap = w.iloc[idx_pos].dropna().sort_values(ascending=False)
        snap = snap[snap > 0.001]  # filter tiny weights
        ax.bar(range(len(snap)), snap.values, color="#1f77b4", alpha=0.75)
        ax.set_xticks(range(len(snap)))
        ax.set_xticklabels(snap.index.tolist(), rotation=90, fontsize=5)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0%}"))
        if col_idx == 0:
            ax.set_ylabel(strat_label, fontsize=9, fontweight="bold")
        if row_idx == 0:
            ax.set_title(date_label, fontsize=9, fontweight="bold")

fig.suptitle("§2.11 Crisis-Date Allocation Snapshots — 3 Strategies × 3 Dates",
             fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()

---
## Cross-Strategy Structure

Are these 62 strategies distinct bets, or a few risk factors wearing different labels? The full return-correlation matrix reveals clustering: strategies that share factor exposures will also share drawdowns, meaning the 62 rows in the table do not represent 62 independent sources of return.

## §2.12 62×62 Strategy Correlation Matrix

In [ ]:
from IPython.display import Image
Image("../docs/figures/strategy_correlation_matrix.png")

---
## Robustness

Every Sharpe ratio is one draw from a noisy process with fat tails and regime structure. The block bootstrap below quantifies how wide the credible range is around the headline numbers; §2.14 then confirms which pairwise contrasts from Notebook 01 §5 are robust to that uncertainty.

## §2.13 Bootstrap Sharpe CIs

Block bootstrap (block size = 252 days, 10,000 resamples) for the top-10 strategies by Sharpe.
Forest plot with point estimates and 95% confidence intervals.

In [ ]:
TRADING_DAYS = 252
np.random.seed(42)

N_BOOT   = 10_000
BLOCK_SZ = 252

def block_bootstrap_sharpe(s, n_boot=N_BOOT, block_size=BLOCK_SZ):
    arr = s.dropna().values
    n   = len(arr)
    n_blocks = int(np.ceil(n / block_size))
    sharpes  = np.empty(n_boot)
    for b in range(n_boot):
        starts = np.random.randint(0, n - block_size + 1, size=n_blocks)
        sample = np.concatenate([arr[st:st+block_size] for st in starts])[:n]
        mu, sigma = sample.mean(), sample.std()
        sharpes[b] = mu / sigma * np.sqrt(TRADING_DAYS) if sigma > 0 else np.nan
    return sharpes

df_all = build_all_stats(base, vmp)
df_all_clean = df_all[df_all["strategy"] != "VMP(GMV(sample))"]
top10_strats = df_all_clean.nlargest(10, "sharpe")[["strategy", "display", "sharpe", "is_vmp"]].reset_index(drop=True)

print("Computing block bootstrap Sharpe CIs for top-10 strategies (may take ~30 seconds)...")
boot_results = []
for _, row in top10_strats.iterrows():
    s = vmp[row.strategy] if row.is_vmp else base[row.strategy]
    dist_boots = block_bootstrap_sharpe(s)
    lo, hi = np.nanpercentile(dist_boots, [2.5, 97.5])
    boot_results.append({"display": row.display, "point": row.sharpe, "lo": lo, "hi": hi})
    print(f"  {row.display:35s}  {row.sharpe:.3f}  [{lo:.3f}, {hi:.3f}]")

df_boot = pd.DataFrame(boot_results).sort_values("point")

fig, ax = plt.subplots(figsize=(9, 7))
for i, row in enumerate(df_boot.itertuples()):
    is_vmp_strat = "VMP" in row.display
    color = "#1f77b4" if is_vmp_strat else "#7f7f7f"
    ax.plot([row.lo, row.hi], [i, i], color=color, lw=2.5, alpha=0.8, solid_capstyle="butt")
    ax.scatter([row.point], [i], color=color, s=60, zorder=5)
    ax.text(row.hi + 0.005, i, f"{row.point:.3f}", va="center", fontsize=8)

ax.set_yticks(range(len(df_boot)))
ax.set_yticklabels(df_boot["display"].tolist(), fontsize=8.5)
ax.axvline(1.0, color="black", ls="--", lw=0.8, alpha=0.5)
ax.text(1.002, len(df_boot)-0.5, "Sharpe=1.0", fontsize=7, color="dimgray")
ax.set_xlabel("Annualized Sharpe Ratio")
ax.set_title(
    f"Block Bootstrap Sharpe 95% CIs — Top-10 Strategies\n"
    f"(block size={BLOCK_SZ} days, {N_BOOT:,} resamples)",
    fontsize=10, fontweight="bold"
)

vmp_patch = mpatches.Patch(color="#1f77b4", label="VMP variant")
base_patch = mpatches.Patch(color="#7f7f7f", label="Base strategy")
ax.legend(handles=[vmp_patch, base_patch], fontsize=8, loc="lower right")
fig.tight_layout()
plt.savefig("../docs/figures/bootstrap_sharpe_cis.png", dpi=150, bbox_inches="tight")
plt.show()

> **Reading — §2.13:** Block bootstrap with 252-day blocks preserves the regime-length autocorrelation that naive t-intervals miss. Wide confidence intervals mean the headline Sharpe is consistent with very different underlying performance — the interval is the credible range, not the point estimate. Bootstrap CIs capture sampling uncertainty along one realized path; model risk and parameter uncertainty through portfolio construction are additional layers not shown. *See §4 Statistical Robustness; Finding R1–R3.*

## §2.14 Pairwise Sharpe-Difference Significance — recap

Full significance tables and derivations live in **Notebook 01, §5**. The headline findings: R1 — MSR(LW) vs sample (Δ=+0.164, p=0.259) is directional but not significant; R3 — SWITCH(v2a) vs SWITCH(LW) (Δ=+0.434, p=0.040) is the study's only clearly significant contrast; R4 — HRP is insensitive to estimator choice (p=0.506). The sign-test for VMP universality (24/24, p≈6×10⁻⁸) is the strongest statistical result in Paper 1. What this chapter uniquely adds: §2.13 bootstrap CIs quantify the sampling uncertainty around those Sharpe points, and **Part 3** confirms that R3 (SWITCH regime skill) and the VMP lift both survive on the true OOS window (2023–2026).

---
## Sensitivity

Do the findings depend on the exact universe and cost assumptions chosen? §2.15 re-runs the comparison including BTC-USD (2008–2026 sub-period) to test survivorship hygiene, and §2.16 replaces the flat 10 bps benchmark with per-asset stratified costs to confirm the TC story is not an artifact of the flat-rate approximation.

## §2.15 BTC Survivorship Sensitivity

**Legacy analysis (2008–2026 sub-period):** BTC-USD first traded on 2010-07-13, leaving 13.8% of
the harness period (636 of 4,611 days) forward-filled at \$0.06. The full study uses the 29-asset
2003–2026 universe (BTC excluded). To quantify survivorship distortion, this section compares the
30-asset with-BTC universe against the 29-asset no-BTC universe over the common **2008–2026
sub-period** — an apples-to-apples comparison using eight representative strategies.

In [ ]:
from aiam.evaluation.switch_assembly import assemble_switch_returns

# 30-asset (with BTC, 2008-2026) vs 29-asset (without BTC, 2008-2026 subset)
SWITCH_V2A_RULE = {0: "MSR(ledoit_wolf)", 5: "MSR(sample)"}

base_legacy = pd.read_parquet(ROOT / 'data/cache/portfolio_returns/24strategies_2008_2026.parquet')
vmp_legacy  = pd.read_parquet(ROOT / 'data/cache/portfolio_returns/24strategies_vmp_2008_2026.parquet')
no_btc = pd.read_parquet(ROOT / 'data/cache/portfolio_returns/8strategies_no_btc_2008_2026.parquet')
regime_sig_legacy = pd.read_parquet(ROOT / "data/cache/regime_signals.parquet")["dominant_regime"].dropna()
switch_v2a_legacy = assemble_switch_returns(
    base_legacy, regime_sig_legacy, SWITCH_V2A_RULE, "MDP(ledoit_wolf)"
).rename("SWITCH(v2a)")

COL_MAP = {
    'EW': 'EW', 'GMV(ledoit_wolf)': 'GMV(LW)', 'MSR(ledoit_wolf)': 'MSR(LW)',
    'MDP(ledoit_wolf)': 'MDP(LW)', 'HRP(ledoit_wolf)': 'HRP(LW)',
    'SWITCH(ledoit_wolf)': 'SWITCH(LW)', 'SWITCH(v2a)': 'SWITCH(v2a)',
    'VMP(MSR(ledoit_wolf))': 'VMP(MSR(LW))',
}

with_btc_s = {
    'EW': ann_sharpe(base_legacy['EW']),
    'GMV(ledoit_wolf)': ann_sharpe(base_legacy['GMV(ledoit_wolf)']),
    'MSR(ledoit_wolf)': ann_sharpe(base_legacy['MSR(ledoit_wolf)']),
    'MDP(ledoit_wolf)': ann_sharpe(base_legacy['MDP(ledoit_wolf)']),
    'HRP(ledoit_wolf)': ann_sharpe(base_legacy['HRP(ledoit_wolf)']),
    'SWITCH(ledoit_wolf)': ann_sharpe(base_legacy['SWITCH(ledoit_wolf)']),
    'SWITCH(v2a)': ann_sharpe(switch_v2a_legacy),
    'VMP(MSR(ledoit_wolf))': ann_sharpe(vmp_legacy['VMP(MSR(ledoit_wolf))']),
}
without_btc_s = {c: ann_sharpe(no_btc[c]) for c in no_btc.columns}
labels = [COL_MAP[k] for k in with_btc_s]
deltas = [with_btc_s[k] - without_btc_s[k] for k in with_btc_s]

x = np.arange(len(labels))
w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.bar(x - w/2, [with_btc_s[k] for k in with_btc_s], w, label='30-asset (with BTC)', color='#1f77b4')
ax.bar(x + w/2, [without_btc_s[k] for k in with_btc_s], w, label='29-asset (no BTC)', color='#ff7f0e')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Sharpe'); ax.set_title('Sharpe: 30-asset (with BTC) vs 29-asset (no BTC)', fontsize=10)
ax.legend(fontsize=8)

ax2 = axes[1]
colors_delta = ['#d62728' if d > 0 else '#1f77b4' for d in deltas]
ax2.bar(x, deltas, color=colors_delta)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=45, ha='right')
ax2.set_ylabel('Sharpe difference (with BTC − without BTC)')
ax2.set_title('BTC Impact on Sharpe (2008–2026, legacy 30-asset universe)', fontsize=10)
ax2.axhline(0, color='black', lw=0.8)
fig.tight_layout()
plt.show()

median_delta = np.median(deltas)
print(f"Median BTC impact on Sharpe: {median_delta:.3f}")
print(f"Mean BTC impact:             {np.mean(deltas):.3f}")
print(f"Positive (BTC helped): {sum(d > 0 for d in deltas)}/{len(deltas)}")

> **Reading — §2.15:** Dropping BTC-USD from the 30-asset universe reduces median Sharpe by 0.229 across eight comparable strategies — well above the 0.02 materiality threshold. MSR(LW) and VMP(MSR(LW)) are most affected (+0.30 and +0.28), reflecting BTC's high-return contribution during trending episodes. HRP(LW) is the exception: Sharpe *improves* by 0.14 without BTC, because BTC's extreme volatility disrupts the correlation-cluster structure that HRP's dendrogram relies on. The headline findings — VMP universally improves, LW shrinkage helps MSR — hold in both universes. The full study uses the 29-asset 2003–2026 universe (BTC excluded) throughout; this legacy sub-period analysis is preserved for reproducibility of Finding 15.

## §2.16 Asset-Class-Stratified Transaction Costs

The flat 10 bps assumption overstates costs for ETF-heavy strategies and understates for BTC-heavy ones. This section applies per-asset costs following §3.3.4 in the paper (Finding 13 update): 2 bps for fixed-income ETFs, 3 bps for broad/sector ETFs, 5 bps for single stocks and international ETFs, 30 bps for BTC.

In [ ]:
from aiam.evaluation.transaction_costs import apply_costs, apply_stratified_costs, STRATIFIED_COST_BPS

def _col_to_stem(col):
    return col.replace("(", "_").replace(")", "")

weights_cache_strat = {}
for _col in base.columns:
    _w = load_weights_cache(_col_to_stem(_col))
    weights_cache_strat[_col] = _w

flat10, strat_ns, names_s, families_s = [], [], [], []
for _col in base.columns:
    for _is_v in [False, True]:
        _vcol = f"VMP({_col})" if _is_v else _col
        _r = vmp[_vcol] if _is_v else base[_col]
        _w = weights_cache_strat.get(_col)
        _flat_n  = ann_sharpe(apply_costs(_r, _w))          if _w is not None else np.nan
        _strat_n = ann_sharpe(apply_stratified_costs(_r, _w)) if _w is not None else np.nan
        _fam = FAMILY_MAP.get(_col, "Classical MV")
        flat10.append(_flat_n);  strat_ns.append(_strat_n)
        names_s.append(_vcol.replace("ledoit_wolf", "LW").replace("(oas)", "(OAS)"))
        families_s.append(_fam)

valid = [(f, s, n, fm) for f, s, n, fm in zip(flat10, strat_ns, names_s, families_s)
         if not (np.isnan(f) or np.isnan(s))]
if valid:
    all_f = [x[0] for x in valid]
    lo_v = max(min(all_f) - 0.05, 0.2)
    hi_v = max(all_f) + 0.05
else:
    lo_v, hi_v = 0.3, 1.55

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([lo_v, hi_v], [lo_v, hi_v], 'k--', lw=1, zorder=0, label='45° line')
for fam, col_c in FAMILY_COLORS.items():
    idx = [i for i, f in enumerate(families_s) if f == fam
           and not (np.isnan(flat10[i]) or np.isnan(strat_ns[i]))]
    if idx:
        ax.scatter([flat10[i] for i in idx], [strat_ns[i] for i in idx],
                   c=col_c, label=fam, alpha=0.7, s=40, zorder=3)
ax.set_xlabel('Net Sharpe (flat 10 bps)')
ax.set_ylabel('Net Sharpe (stratified)')
ax.set_title('Flat vs Stratified Transaction Costs — All 62 Strategy Records')
ax.legend(loc='upper left', fontsize=7)
fig.tight_layout()
plt.savefig("../docs/figures/stratified_vs_flat_costs.png", dpi=150, bbox_inches="tight")
plt.show()

> **Reading — §2.16:** The scatter above the 45° line shows that stratified costs are cheaper than the flat 10 bps benchmark for almost all strategies, because fixed-income ETFs cost only 2 bps and broad equity ETFs only 3 bps. The largest beneficiaries are high-turnover equity strategies: FF3-Mom rises from 0.310 to 0.485 net Sharpe. BTC-heavy strategies (labeled) gain slightly from cheaper equity components; the 30 bps BTC cost matters only where BTC weight is large. The qualitative ranking — VMP regime and diversification strategies lead — survives unchanged under stratified costs.

---
## OOS Holdout Validation — the acid test

Full-sample Sharpes are retrospective; a practitioner wants to know whether the findings survive on data no strategy ever touched. The 2023–2026 window is Paper 1's true OOS: VMP lift and LW shrinkage survive, SWITCH(v2a) delta shrinks to +0.114 (reduced power on 3 years), and the ranking order is broadly preserved. This section is the bridge to Paper 2: learned approaches are evaluated against this same OOS bar.

## Part 3: OOS Holdout Validation

In [ ]:
from IPython.display import display

TRAIN_END = "2022-12-31"
TEST_START = "2023-01-01"

print("=== OOS Columns ===")
print(sw_oos.columns.tolist())

print("\n=== Training vs Test Period Sharpe ===")
oos_rows = []
for col in sw_oos.columns:
    r_full = sw_oos[col].reindex(base.index).dropna()
    r_train = r_full.loc[:TRAIN_END]
    r_test  = r_full.loc[TEST_START:]
    oos_rows.append({
        "Column": col,
        "Full Sharpe":  round(ann_sharpe(r_full), 4) if len(r_full) > 21 else np.nan,
        "Train Sharpe": round(ann_sharpe(r_train), 4) if len(r_train) > 21 else np.nan,
        "Test Sharpe":  round(ann_sharpe(r_test), 4) if len(r_test) > 21 else np.nan,
        "N train": len(r_train),
        "N test":  len(r_test),
    })

df_oos = pd.DataFrame(oos_rows)
display(df_oos)

# Top 10 by test-period Sharpe
print("\n=== Top 10 by Test-Period Sharpe ===")
all_test_rets = pd.concat([
    base.loc[TEST_START:],
    vmp.loc[TEST_START:]
], axis=1)
test_sharpes = all_test_rets.apply(lambda c: ann_sharpe(c.dropna()) if len(c.dropna()) > 21 else np.nan)
top10_test = test_sharpes.nlargest(10)
df_top10 = pd.DataFrame({
    "strategy": top10_test.index,
    "display": [s.replace("ledoit_wolf", "LW") for s in top10_test.index],
    "test_sharpe": top10_test.values.round(4),
})
display(df_top10)